# Dropout Prevention - Preprocess
This notebook is a separated copy of the original dropout workflow. It handles data loading, early feature engineering, data cleaning, exploratory checks, and saving a processed dataset for the next steps.

**Output of this notebook:** `../../data/processed/dropout/dropout_preprocessed.csv`


## Import Libraries

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


## Load OULA Datasets

In [12]:
base_dir = Path.cwd().resolve()
if not (base_dir / "data").exists():
    base_dir = base_dir.parent.parent

raw_dir = base_dir / "data" / "raw" / "oula"

info = pd.read_csv(raw_dir / "studentInfo.csv")
vle = pd.read_csv(raw_dir / "studentVle.csv")
student_assess = pd.read_csv(raw_dir / "studentAssessment.csv")
assess_meta = pd.read_csv(raw_dir / "assessments.csv")

print('studentInfo:', info.shape)
print('studentVle:', vle.shape)
print('studentAssessment:', student_assess.shape)
print('assessments:', assess_meta.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/admin/Documents/GitHub/STUDv1.0/Student Dropout Prediction and Alert System/notebooks/dropout/data/oula/studentInfo.csv'

## Early Feature Engineering
The model is designed as an early warning system, so only the first 40 days of learning activity are used.


In [ ]:
EARLY_THRESHOLD = 40

# Behavioural features from early VLE activity
early_vle = vle[vle['date'] <= EARLY_THRESHOLD]

vle_features = early_vle.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    avg_clicks_per_day=('sum_click', 'mean')
).reset_index()

course_avg = vle_features.groupby(['code_module', 'code_presentation'])['total_clicks'].transform('mean')
course_std = vle_features.groupby(['code_module', 'code_presentation'])['total_clicks'].transform('std')
vle_features['relative_engagement'] = (vle_features['total_clicks'] - course_avg) / course_std

# Academic features from early assessments
student_assess['score'] = pd.to_numeric(student_assess['score'], errors='coerce')
student_assess['date_submitted'] = pd.to_numeric(student_assess['date_submitted'], errors='coerce')
assess_meta['date'] = pd.to_numeric(assess_meta['date'], errors='coerce')

assess_merge = student_assess.merge(assess_meta, on='id_assessment')
early_assess = assess_merge[assess_merge['date'] <= EARLY_THRESHOLD].copy()
early_assess['days_late'] = early_assess['date_submitted'] - early_assess['date']

assess_features = early_assess.groupby('id_student').agg(
    avg_score=('score', 'mean'),
    avg_lateness=('days_late', 'mean')
).reset_index()

display(vle_features.head())
display(assess_features.head())

## Merge and Build the Target

In [3]:
info['target'] = info['final_result'].apply(lambda x: 1 if x in ['Fail', 'Withdrawn'] else 0)

df = info.merge(vle_features, on=['id_student', 'code_module', 'code_presentation'], how='left')
df = df.merge(assess_features, on='id_student', how='left')

print('Merged Dataset shape:', df.shape)
display(df.head())

NameError: name 'info' is not defined

## Exploratory Checks and Missing Values

In [4]:
display(df.describe())
df.info()

NameError: name 'df' is not defined

In [5]:
print('Duplicate rows:', df.duplicated().sum())
print('Missing values by column:')
print(df.isnull().sum())

plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()


NameError: name 'df' is not defined

## Fill Missing Values and Save the Processed Dataset

In [6]:
df.fillna({
    'total_clicks': 0,
    'active_days': 0,
    'avg_clicks_per_day': 0,
    'relative_engagement': -1,
    'avg_score': 0,
    'avg_lateness': 0
}, inplace=True)

features = [
    'total_clicks',
    'active_days',
    'relative_engagement',
    'avg_score',
    'avg_lateness',
    'num_of_prev_attempts',
    'studied_credits'
]

processed_path = base_dir / "data" / "processed" / "dropout" / "dropout_preprocessed.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(processed_path, index=False)

print('Remaining missing values:', df.isnull().sum().sum())
print('Saved processed dataset to:', processed_path)
print('Feature columns:', features)


<>:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
/var/folders/b3/0fd8bgg112d1_6p01j251slr0000gn/T/ipykernel_42301/78298047.py:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  processed_path = '..\data\processed\dropout_preprocessed.csv'
/var/folders/b3/0fd8bgg112d1_6p01j251slr0000gn/T/ipykernel_42301/78298047.py:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  processed_path = '..\data\processed\dropout_preprocessed.csv'


NameError: name 'df' is not defined